# U9-03 · Caso práctico (3): evaluación, futuro y despliegue

*Especialista en Inteligencia Artificial (IFCD107) · Unidad 9*

Ejecuta cada celda con **Mayús + Intro** (en Colab, también con ▶). Con GPU (*Entorno de ejecución → Cambiar tipo de entorno de ejecución*) las redes se entrenan mucho más deprisa.

> Datos: Moro, S., Rita, P. y Cortez, P. (2014). *Bank Marketing*. UCI, CC BY 4.0. https://doi.org/10.24432/C5K306 · Moro, S., Cortez, P. y Rita, P. (2014). *Decision Support Systems*, 62, 22–31. https://doi.org/10.1016/j.dss.2014.03.001

In [ ]:
%pip install -q fastapi httpx
# Descarga los datos y módulos auxiliares (solo la primera vez)
import os, urllib.request
for f in ["bank-additional-full.csv", "banco.py", "evaluar.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve("https://machinelearning-formacion.github.io/ifcd107/recursos/datos/u9/" + f, f)

## 9.5 · Evaluación orientada al negocio

`z1_ganancias.py`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from banco import X_e, X_p, y_e, y_p
from evaluar import prep_lineal, potenciacion

modelos = {"potenciación": potenciacion(),
           "logística": make_pipeline(
               prep_lineal(), LogisticRegression(max_iter=3000))}
fig, ej = plt.subplots(figsize=(3.4, 2.6), layout="constrained")
y = y_p.to_numpy()
for n, m in modelos.items():
    p = m.fit(X_e, y_e).predict_proba(X_p)[:, 1]
    capt = np.cumsum(y[np.argsort(-p)]) / y.sum()   # % capturado
    f = np.arange(1, len(y) + 1) / len(y)
    ej.plot(100 * f, 100 * capt, label=n)
    print(f"{n:<13}" + "".join(
        f"  {k}% -> {capt[int(k / 100 * len(y)) - 1]:.1%}"
        for k in [10, 20, 30, 50]))
ej.plot([0, 100], [0, 100], color="gray", ls=":", label="al azar")
ej.set_xlabel("% de clientes llamados")
ej.set_ylabel("% de suscriptores alcanzados")
ej.legend(fontsize=6); plt.show()

`z0_curvas.py`

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from banco import X_e, X_p, y_e, y_p
from evaluar import prep_lineal, potenciacion

fig, ej = plt.subplots(1, 2, figsize=(4.6, 2.2), layout="constrained")
for n, m in [("potenciación", potenciacion()),
             ("logística", make_pipeline(
                 prep_lineal(), LogisticRegression(max_iter=3000)))]:
    m.fit(X_e, y_e)
    RocCurveDisplay.from_estimator(m, X_p, y_p, ax=ej[0], name=n)
    PrecisionRecallDisplay.from_estimator(m, X_p, y_p, ax=ej[1],
                                          name=n)
ej[0].plot([0, 1], [0, 1], ls=":", color="gray")
ej[1].axhline(y_p.mean(), ls=":", color="gray")
ej[0].set_xlabel("tasa de falsos positivos")
ej[0].set_ylabel("sensibilidad")
ej[1].set_xlabel("sensibilidad"); ej[1].set_ylabel("precisión")
for e in ej:
    e.legend(fontsize=5)
plt.show()
print("figura guardada")

`z10_bootstrap.py`

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score
from banco import X_e, X_p, y_e, y_p
from evaluar import potenciacion, mejora

p = potenciacion().fit(X_e, y_e).predict_proba(X_p)[:, 1]
y = y_p.to_numpy()
rng = np.random.default_rng(0)
auc, lift = [], []
for _ in range(1000):                     # remuestrear la prueba
    i = rng.integers(0, len(y), len(y))
    auc.append(roc_auc_score(y[i], p[i]))
    lift.append(mejora(y[i], p[i]))
for n, v in [("AUC", auc), ("lift@10%", lift)]:
    lo, hi = np.percentile(v, [2.5, 97.5])
    print(f"{n:<9} intervalo del 95 %: {lo:.3f} a {hi:.3f}")

`z2_beneficio.py`

In [ ]:
import numpy as np
from banco import X_e, X_p, y_e, y_p
from evaluar import potenciacion

COSTE, BENEFICIO = 5.0, 50.0      # supuestos ilustrativos, en euros
p = potenciacion().fit(X_e, y_e).predict_proba(X_p)[:, 1]
y = y_p.to_numpy()[np.argsort(-p)]
neto = np.cumsum(BENEFICIO * y - COSTE)  # llamando a los k primeros
k = neto.argmax() + 1
print(f"llamar a todos: {neto[-1]:9.0f} €")
print(f"llamar a los {k} mejores ({k / len(y):.1%}):"
      f" {neto[k-1]:9.0f} €, con {y[:k].sum()} de {y.sum()}"
      f" suscriptores")
umbral = COSTE / BENEFICIO        # llamar si p * B > C
print(f"regla p > {umbral}: se llamaría a {(p > umbral).sum()}"
      f" clientes")
for c in [2.0, 10.0]:
    n = np.cumsum(BENEFICIO * y - c)
    print(f"con un coste de {c:4.1f} €: llamar a {n.argmax() + 1}"
          f" ({(n.argmax() + 1) / len(y):.1%}), {n.max():8.0f} €")

`z3_calibracion.py`

In [ ]:
import numpy as np
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
from banco import X_e, X_p, y_e, y_p
from evaluar import potenciacion

for peso in [None, "balanced"]:
    p = potenciacion(class_weight=peso).fit(X_e, y_e) \
        .predict_proba(X_p)[:, 1]
    real, prev = calibration_curve(y_p, p, n_bins=5,
                                   strategy="quantile")
    b = brier_score_loss(y_p, p)
    print(f"pesos {str(peso):<9} Brier {b:.4f}")
    print("   previsto:", prev.round(3))
    print("   real:    ", real.round(3))

`z4_importancia.py`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.inspection import (permutation_importance,
                                PartialDependenceDisplay)
from banco import X_e, X_p, y_e, y_p
from evaluar import potenciacion

m = potenciacion().fit(X_e, y_e)
r = permutation_importance(m, X_p, y_p, scoring="average_precision",
                           n_repeats=5, random_state=0)
imp = pd.Series(r.importances_mean, X_p.columns).sort_values(
    ascending=False)
print("pérdida de AP al barajar cada variable:")
print(imp.head(8).round(4).to_string())
print("variables con efecto nulo o negativo:", (imp <= 0).sum())
fig, ej = plt.subplots(1, 2, figsize=(4.6, 1.9), sharey=True,
                       layout="constrained")
Xf = X_e.astype({"age": float})           # exige números reales
PartialDependenceDisplay.from_estimator(
    m, Xf, ["emp.var.rate", "age"], ax=ej, subsample=3000,
    response_method="predict_proba", random_state=0)
ej[0].set_xlabel("variación del empleo"); ej[1].set_xlabel("edad")
ej[0].set_ylabel("probabilidad de suscribir"); ej[1].set_ylabel("")
plt.show()

`z11_un_cliente.py`

In [ ]:
import pandas as pd
from banco import X_e, X_p, y_e
from evaluar import potenciacion

m = potenciacion().fit(X_e, y_e)
p = m.predict_proba(X_p)[:, 1]
c = X_p.iloc[[p.argmax()]]                # el cliente más prometedor
print(c[["age", "job", "contact", "month", "poutcome", "pdays",
         "euribor3m"]].to_string(index=False))
print(f"probabilidad: {p.max():.3f}")
cambios = {"contact": "telephone", "poutcome": "nonexistent",
           "pdays": 999, "month": "may", "age": 40}
for v, nuevo in cambios.items():          # ¿y si fuera distinto?
    otro = c.assign(**{v: nuevo})
    txt = f"{v} = {nuevo}"
    print(f"  si {txt:<22} -> {m.predict_proba(otro)[0, 1]:.3f}")

`z5_equidad.py`

In [ ]:
import numpy as np
import pandas as pd
from banco import X_e, X_p, y_e, y_p
from evaluar import potenciacion

p = potenciacion().fit(X_e, y_e).predict_proba(X_p)[:, 1]
lista = p >= np.quantile(p, 0.8)          # se llama al 20 % mejor
g = pd.cut(X_p.age, [16, 25, 40, 60, 100])
r = pd.DataFrame({"grupo": g, "y": y_p, "lista": lista})
t = r.groupby("grupo", observed=True).apply(lambda d: pd.Series({
    "n": len(d), "% suscribe": 100 * d.y.mean(),
    "% en la lista": 100 * d.lista.mean(),
    "sensibilidad": d.lista[d.y == 1].mean()}), include_groups=False)
print(t.round(2).to_string())

## 9.6 · El futuro: cuando el mundo cambia

`z6_futuro.py`

In [ ]:
from scipy.stats import ks_2samp
from banco import X_e, y_e, X_f, y_f, X_p, y_p
from evaluar import potenciacion, metricas

m = potenciacion().fit(X_e, y_e)
for nombre, X, y in [("prueba 2008-09", X_p, y_p),
                     ("futuro 2010   ", X_f, y_f)]:
    p = m.predict_proba(X)[:, 1]
    print(f"{nombre}: {metricas(y, p)}")
    print(f"   probabilidad media {p.mean():.3f},"
          f" real {y.mean():.3f}")
for v in ["euribor3m", "age", "campaign"]:
    r = ks_2samp(X_e[v], X_f[v])
    print(f"{v:<10} KS {r.statistic:.3f}")

`z12_vigilancia.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from banco import d, X, y, X_e, y_e
from evaluar import potenciacion

m = potenciacion().fit(X_e, y_e)
f = (d.año == 2010).to_numpy()
r = pd.DataFrame({"mes": d.month[f], "y": y[f],
                  "p": m.predict_proba(X[f])[:, 1]})
orden = ["mar", "apr", "may", "jun", "jul", "aug", "sep", "oct",
         "nov"]
print("mes  llamadas  real  previsto  AUC   alerta")
for mes in orden:
    g = r[r.mes == mes]
    auc = roc_auc_score(g.y, g.p)
    alerta = "calibración" if abs(g.y.mean() - g.p.mean()) > 0.1 \
        else ""
    print(f"{mes}  {len(g):8d}  {g.y.mean():.2f}  {g.p.mean():8.2f}"
          f"  {auc:.2f}  {alerta}")

`z7_reentrenar.py`

In [ ]:
import pandas as pd
from banco import d, X, y, X_e, y_e
from evaluar import potenciacion, metricas

f = d.año == 2010
temprano = f & d.month.isin(["mar", "apr", "may", "jun"])
tardio = f & ~temprano
print("2010 temprano:", temprano.sum(),
      "| 2010 tardío:", tardio.sum())
viejo = potenciacion().fit(X_e, y_e)
nuevo = potenciacion().fit(pd.concat([X_e, X[temprano]]),
                           pd.concat([y_e, y[temprano]]))
for n, m in [("sin reentrenar", viejo), ("reentrenado  ", nuevo)]:
    p = m.predict_proba(X[tardio])[:, 1]
    print(f"{n}: {metricas(y[tardio], p)}")
    print(f"   prob. media {p.mean():.3f}"
          f" (real {y[tardio].mean():.3f})")

`z15_recalibrar.py`

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
from banco import d, X, y, X_e, y_e
from evaluar import potenciacion

m = potenciacion().fit(X_e, y_e)
f = d.año == 2010
marzo, resto = f & (d.month == "mar"), f & (d.month != "mar")
logit = lambda p: np.log(p / (1 - p)).reshape(-1, 1)
p_mar = m.predict_proba(X[marzo])[:, 1]
platt = LogisticRegression().fit(logit(p_mar), y[marzo])  # Platt
p = m.predict_proba(X[resto])[:, 1]
q = platt.predict_proba(logit(p))[:, 1]
print(f"llamadas de marzo para recalibrar: {marzo.sum()}")
print(f"abril a noviembre: real {y[resto].mean():.3f}")
for n, v in [("sin recalibrar", p), ("recalibrado", q)]:
    print(f"  {n:<15} prob. media {v.mean():.3f},"
          f" Brier {brier_score_loss(y[resto], v):.4f},"
          f" llamar (p > 0,1): {(v > 0.1).mean():.1%}")

`z16_recientes.py`

In [ ]:
import numpy as np
import pandas as pd
from banco import d, X, y
from evaluar import potenciacion, metricas

ent = (d.año <= 2009) | ((d.año == 2010) & d.month.isin(
    ["mar", "apr", "may", "jun"]))
pru = (d.año == 2010) & ~ent
edad_dato = np.where(d.año == 2010, 0, np.where(d.año == 2009, 1, 2))
for n, w in [("todos igual", np.ones(len(d))),
             ("más peso a lo reciente", 4.0 ** -edad_dato)]:
    peso = {"histgradientboostingclassifier__sample_weight": w[ent]}
    m = potenciacion().fit(X[ent], y[ent], **peso)
    p = m.predict_proba(X[pru])[:, 1]
    print(f"{n:<23} {metricas(y[pru], p)}")
    print(f"   prob. media {p.mean():.3f} (real {y[pru].mean():.3f})")

## 9.7 · Despliegue

`z8_desplegar.py`

In [ ]:
import sqlite3
import joblib
import pandas as pd
from banco import X, y, d
from evaluar import potenciacion

hist = d.año <= 2009
m = potenciacion().fit(X[hist], y[hist])   # todo lo disponible
joblib.dump(m, "campana.joblib")
nuevos = X[~hist]                          # clientes a puntuar
con = sqlite3.connect("campana.db")
nuevos.reset_index(names="cliente").to_sql(
    "clientes", con, if_exists="replace", index=False)
lote = pd.read_sql("SELECT * FROM clientes", con)
cargado = joblib.load("campana.joblib")
lote["prob"] = cargado.predict_proba(lote[X.columns])[:, 1].round(4)
lote[["cliente", "prob"]].to_sql("puntuaciones", con,
                                 if_exists="replace", index=False)
print(pd.read_sql("""SELECT cliente, prob FROM puntuaciones
                     ORDER BY prob DESC LIMIT 5""", con).to_string(
    index=False))
print("clientes puntuados:", len(lote))

`z13_api.py`

In [ ]:
import joblib
import pandas as pd
from typing import Literal
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from banco import X_f

modelo = joblib.load("campana.joblib")
app = FastAPI(title="Prioridad de llamada")
SiNo = Literal["no", "yes", "unknown"]

class Cliente(BaseModel):                 # lo que se acepta
    age: int = Field(ge=17, le=100)
    job: str
    marital: str
    education: str
    default: SiNo
    housing: SiNo
    loan: SiNo
    contact: Literal["cellular", "telephone"]
    month: str
    day_of_week: Literal["mon", "tue", "wed", "thu", "fri"]
    campaign: int = Field(ge=1)
    pdays: int
    previous: int = Field(ge=0)
    poutcome: Literal["failure", "nonexistent", "success"]
    emp_var_rate: float = Field(alias="emp.var.rate")
    cons_price_idx: float = Field(alias="cons.price.idx")
    cons_conf_idx: float = Field(alias="cons.conf.idx")
    euribor3m: float
    nr_employed: float = Field(alias="nr.employed")

@app.post("/prioridad")
def prioridad(c: Cliente):
    x = pd.DataFrame([c.model_dump(by_alias=True)])
    p = float(modelo.predict_proba(x)[0, 1])
    return {"probabilidad": round(p, 3), "llamar": p > 0.1}

cliente = TestClient(app)
ejemplo = X_f.iloc[0].to_dict()           # un cliente real de 2010
print({k: ejemplo[k] for k in ["age", "job", "contact", "month"]})
print(cliente.post("/prioridad", json=ejemplo).json())
r = cliente.post("/prioridad", json=dict(ejemplo, contact="fax"))
print(r.status_code, r.json()["detail"][0]["msg"])

`z14_pruebas.py`

In [ ]:
import joblib
import numpy as np
from sklearn.metrics import roc_auc_score
from banco import X_p, y_p

m = joblib.load("campana.joblib")
def prueba(nombre, condicion):
    print(f"{'OK   ' if condicion else 'FALLA'} {nombre}")
cols = list(m[0].feature_names_in_)
prueba("no usa la duración de la llamada", "duration" not in cols)
p = m.predict_proba(X_p)[:, 1]
prueba("probabilidades entre 0 y 1", ((p >= 0) & (p <= 1)).all())
nuevo = X_p.head(3).assign(job="astronauta")   # categoría nueva
prueba("acepta categorías desconocidas",
       np.isfinite(m.predict_proba(nuevo)).all())
prueba("AUC de referencia por encima de 0,7",
       roc_auc_score(y_p, p) > 0.7)
a, b = X_p.head(1), X_p.head(1).assign(campaign=30)
prueba("insistir mucho no aumenta la probabilidad",
       m.predict_proba(b)[0, 1] <= m.predict_proba(a)[0, 1])

`z17_ficha.py`

In [ ]:
import json
import hashlib
import joblib
import sklearn
from sklearn.metrics import roc_auc_score, average_precision_score
from banco import X_p, y_p
from evaluar import mejora

m = joblib.load("campana.joblib")
p = m.predict_proba(X_p)[:, 1]
ficha = {
    "nombre": "prioridad-deposito-plazo",
    "uso_previsto": "ordenar candidatos de campañas de telemarketing",
    "usos_no_permitidos": ["decisiones de crédito",
                           "otros productos"],
    "datos": "bank-additional-full.csv, 2008-2009, sin 'duration'",
    "licencia_datos": "CC BY 4.0",
    "variables": list(m[0].feature_names_in_),
    "algoritmo": type(m[-1]).__name__,
    "scikit_learn": sklearn.__version__,
    "sha256": hashlib.sha256(open("campana.joblib", "rb").read())
              .hexdigest()[:16],
    "metricas_prueba": {"auc": 0.766, "ap": 0.338, "lift10": 4.40},
    "en_datos_vistos": {"auc": round(roc_auc_score(y_p, p), 3),
                     "ap": round(average_precision_score(y_p, p), 3),
                     "lift10": round(float(mejora(y_p, p)), 2)},
    "limitaciones": "se degrada si cambia el contexto económico",
    "vigilancia": "mensual: calibración, AUC, deciles"}
json.dump(ficha, open("ficha.json", "w"), ensure_ascii=False,
          indent=1)
for k in ["nombre", "algoritmo", "scikit_learn", "metricas_prueba",
          "en_datos_vistos"]:
    print(f"{k}: {ficha[k]}")
print("variables:", len(ficha["variables"]))

`z9_informe.py`

In [ ]:
import numpy as np
import pandas as pd
from banco import X_e, X_p, y_e, y_p
from evaluar import potenciacion

p = potenciacion().fit(X_e, y_e).predict_proba(X_p)[:, 1]
dec = pd.qcut(-p, 10, labels=range(1, 11))   # decil 1 = mejores
t = pd.DataFrame({"decil": dec, "y": y_p.to_numpy(), "p": p}) \
    .groupby("decil", observed=True).agg(
        clientes=("y", "size"), suscriben=("y", "sum"),
        tasa=("y", "mean"), prob_media=("p", "mean"))
t["% acumulado"] = 100 * t.suscriben.cumsum() / t.suscriben.sum()
t["tasa"] = (100 * t.tasa).round(1)
print(t.round({"prob_media": 3, "% acumulado": 1}).to_string())

**Prueba.** En `z2_beneficio.py`, cambia el beneficio por depósito a 30 y a 100 euros. ¿A cuántos clientes conviene llamar?

**Ejercicio.** Recalibra con marzo y abril de 2010 en `z15_recalibrar.py`. ¿Mejora la puntuación de Brier?